# T33 - 信用债规模

## 第4章：数据处理与特征工程

---

### 数据处理流程

本章节实现规模统计的核心逻辑，包括：
1. 多维度规模统计（大类、省、市、行业、评级、久期）
2. 数据汇总与存储

In [ ]:
# 导入依赖
import pandas as pd
import sqlalchemy
from sqlalchemy import text, create_engine
from datetime import datetime
import sqlalchemy.pool as pool
import os

# 从环境变量获取配置
MYSQL_HOST = os.environ.get('MYSQL_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
MYSQL_PORT = os.environ.get('MYSQL_PORT', '3306')
MYSQL_USER = os.environ.get('MYSQL_USER', 'hz_work')
MYSQL_PASSWORD = os.environ.get('MYSQL_PASSWORD', '')
MYSQL_DATABASE = os.environ.get('MYSQL_DATABASE', 'bond')

# 创建数据库连接
connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
sql_engine = create_engine(connection_string, poolclass=pool.NullPool)

def log_progress(message):
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{current_time}] {message}")

log_progress("数据库连接创建成功")

### 获取待处理日期列表

In [ ]:
def get_unprocessed_scale_dates():
    """获取需要统计规模的日期列表"""
    sql = """
    SELECT DISTINCT dt FROM bond.marketinfo3
    WHERE dt NOT IN (SELECT DISTINCT dt FROM bond.信用债规模)
    """
    with sql_engine.connect() as conn:
        result = pd.read_sql(sql, conn)
    return result['dt'].tolist()

dt_list = get_unprocessed_scale_dates()
log_progress(f"待处理规模统计日期数量: {len(dt_list)}")
if dt_list:
    log_progress(f"日期范围: {min(dt_list)} ~ {max(dt_list)}")

### 规模统计查询定义

定义15个不同维度的规模统计查询

In [ ]:
def build_scale_queries(dt_list):
    """构建规模统计查询列表"""
    dt_tuple = tuple(dt_list) if len(dt_list) > 1 else f"('{dt_list[0]}')"
    
    queries = [
        # 1. 全部债券汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            DT, 
            '全部' as 分类,
            '' AS 子类, 
            '大类' as 分类方式,
            SUM(ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3
        WHERE 久期 IS NOT NULL
        AND DT IN {dt_tuple}
        GROUP BY DT
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 2. 按大类汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            B.大类 as 分类,
            '' AS 子类, 
            '大类' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        WHERE A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, B.大类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 3. 城投债按省份汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            '城投' as 分类, 
            C.省 AS 子类,
            '省' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_xzqh_ct C ON B.子类 = C.`城投区域`
        WHERE B.大类 = '城投' AND C.`省` IS NOT NULL AND C.`省` != ''
        AND A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, C.省
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 4. 城投债按城市汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            '城投' as 分类, 
            C.市 AS 子类, 
            '市' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_xzqh_ct C ON B.子类 = C.`城投区域`
        WHERE B.大类 = '城投' AND C.`市` IS NOT NULL AND C.`市` != ''
        AND A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, C.市
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 5. 产业债按申万一级汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            '产业' as 分类, 
            C.申万一级 AS 子类,
            '申万一级' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_industrytype1 C ON B.子类 = C.申万行业
        WHERE B.大类 = '产业' AND C.`申万一级` IS NOT NULL AND C.`申万一级` != ''
        AND A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, C.申万一级
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 6. 产业债按申万行业汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            '产业' as 分类, 
            B.子类,
            '申万行业' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        WHERE B.大类 = '产业'
        AND A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, B.子类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 7. 产业债按一级分类汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            '产业' as 分类, 
            C.一级分类 AS 子类,
            '一级分类' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_industrytype1 C ON B.子类 = C.申万行业
        WHERE B.大类 = '产业' AND C.`一级分类` IS NOT NULL AND C.`一级分类` != ''
        AND A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, C.一级分类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 8. 产业债按二级分类汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            '产业' as 分类, 
            C.二级分类 AS 子类,
            '二级分类' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_industrytype1 C ON B.子类 = C.申万行业
        WHERE B.大类 = '产业' AND C.`二级分类` IS NOT NULL AND C.`二级分类` != ''
        AND A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, C.二级分类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 9. 金融债按机构类型汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            '金融' as 分类,
            B.子类 AS 子类,
            '金融机构' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        WHERE B.`大类` = '金融' AND B.子类 IS NOT NULL AND B.子类 != ''
        AND A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, B.子类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 10. ABS按资产类型汇总
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            'ABS' as 分类,
            B.子类 AS 子类,
            '资产' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        WHERE B.大类 = 'ABS' AND B.子类 IS NOT NULL AND B.子类 != ''
        AND A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, B.子类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 11. 按评级汇总（全部）
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            '全部' as 分类,
            B.外评 AS 子类, 
            '评级' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        WHERE A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, B.外评
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 12. 按评级汇总（分大类）
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            B.`大类` as 分类,
            B.外评 AS 子类, 
            '评级' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.basicinfo_all B ON A.trade_code = B.trade_code
        WHERE A.久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, B.`大类`, B.外评
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 13. 按久期汇总（全部）
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            DT, 
            '全部' as 分类,
            CASE 
                WHEN 久期 < 0.75 THEN 0.5
                WHEN 久期 < 1.5 THEN 1
                WHEN 久期 < 2.5 THEN 2
                WHEN 久期 < 3.5 THEN 3
                WHEN 久期 < 4.5 THEN 4
                WHEN 久期 < 6 THEN 5
                WHEN 久期 < 8 THEN 7
                ELSE 10 
            END AS 子类1, 
            '久期' as 分类方式,
            SUM(ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3
        WHERE 久期 IS NOT NULL
        AND DT IN {dt_tuple}
        GROUP BY DT, 子类1
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 14. 按久期汇总（分大类）
        f"""
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT 
            A.DT, 
            B.`大类` as 分类,
            CASE 
                WHEN 久期 < 0.75 THEN 0.5
                WHEN 久期 < 1.5 THEN 1
                WHEN 久期 < 2.5 THEN 2
                WHEN 久期 < 3.5 THEN 3
                WHEN 久期 < 4.5 THEN 4
                WHEN 久期 < 6 THEN 5
                WHEN 久期 < 8 THEN 7
                ELSE 10 
            END AS 子类1, 
            '久期' as 分类方式,
            SUM(A.ths_bond_balance_bond) as 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        WHERE 久期 IS NOT NULL
        AND A.DT IN {dt_tuple}
        GROUP BY A.DT, B.`大类`, 子类1
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """
    ]
    
    return queries

print("规模统计查询定义完成")

### 执行规模统计

In [ ]:
def execute_scale_statistics(dt_list):
    """执行规模统计"""
    if not dt_list:
        log_progress("没有新的数据需要处理")
        return
    
    queries = build_scale_queries(dt_list)
    
    log_progress(f"开始执行 {len(queries)} 个规模统计查询...")
    
    for i, query in enumerate(queries, 1):
        log_progress(f"执行规模统计查询 {i}/{len(queries)}")
        try:
            with sql_engine.connect() as conn:
                conn.execute(text(query))
                conn.commit()
        except Exception as e:
            log_progress(f"查询 {i} 执行失败: {e}")
    
    log_progress("完成规模统计")

execute_scale_statistics(dt_list)

### 清理临时数据

In [ ]:
def cleanup_temp_data():
    """清理临时数据"""
    sql = "DELETE FROM bond.信用债规模 WHERE DT = '2023-6-25'"
    with sql_engine.connect() as conn:
        conn.execute(text(sql))
        conn.commit()
    log_progress("完成临时数据清理")

cleanup_temp_data()

### 数据验证

In [ ]:
# 验证规模统计数据
sql_check = """
SELECT 
    分类方式,
    分类,
    COUNT(DISTINCT DT) as distinct_dates,
    COUNT(DISTINCT 子类) as distinct_subtypes,
    SUM(余额) as total_balance
FROM bond.信用债规模
WHERE DT >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
GROUP BY 分类方式, 分类
ORDER BY 分类方式, 分类
    """

with sql_engine.connect() as conn:
    result = pd.read_sql(sql_check, conn)

print("最近30天规模统计汇总:")
print(result.to_string())

In [ ]:
# 查看最新一天的规模统计
sql_latest = """
SELECT 
    DT,
    分类方式,
    分类,
    子类,
    余额
FROM bond.信用债规模
WHERE DT = (SELECT MAX(DT) FROM bond.信用债规模)
ORDER BY 分类方式, 分类, 余额 DESC
LIMIT 50
    """

with sql_engine.connect() as conn:
    result = pd.read_sql(sql_latest, conn)

print("最新一天规模统计详情:")
print(result.to_string())

### 数据处理工具函数

In [ ]:
def get_scale_by_category(dt, category_type, category=None):
    """
    获取指定日期和分类方式的规模数据
    
    Parameters:
    -----------
    dt : str or date
        查询日期
    category_type : str
        分类方式（大类/省/市/申万一级/评级/久期等）
    category : str, optional
        分类名称
    
    Returns:
    --------
    DataFrame
    """
    sql = """
    SELECT 分类, 子类, 余额
    FROM bond.信用债规模
    WHERE DT = :dt AND 分类方式 = :category_type
    """
    
    params = {'dt': str(dt), 'category_type': category_type}
    
    if category:
        sql += " AND 分类 = :category"
        params['category'] = category
    
    sql += " ORDER BY 余额 DESC"
    
    with sql_engine.connect() as conn:
        result = pd.read_sql(text(sql), conn, params=params)
    
    return result

# 示例：获取最新日期的大类规模
with sql_engine.connect() as conn:
    latest_dt = pd.read_sql("SELECT MAX(DT) as max_dt FROM bond.信用债规模", conn)
    latest_date = latest_dt['max_dt'].iloc[0]

print(f"最新数据日期: {latest_date}")
print("\n大类规模:")
print(get_scale_by_category(latest_date, '大类'))

In [ ]:
def get_scale_trend(category_type, category, subtype=None, start_date=None, end_date=None):
    """
    获取规模趋势数据
    
    Parameters:
    -----------
    category_type : str
        分类方式
    category : str
        分类名称
    subtype : str, optional
        子类名称
    start_date : str, optional
        开始日期
    end_date : str, optional
        结束日期
    
    Returns:
    --------
    DataFrame
    """
    sql = """
    SELECT DT, 余额
    FROM bond.信用债规模
    WHERE 分类方式 = :category_type AND 分类 = :category
    """
    
    params = {'category_type': category_type, 'category': category}
    
    if subtype:
        sql += " AND 子类 = :subtype"
        params['subtype'] = subtype
    
    if start_date:
        sql += " AND DT >= :start_date"
        params['start_date'] = str(start_date)
    
    if end_date:
        sql += " AND DT <= :end_date"
        params['end_date'] = str(end_date)
    
    sql += " ORDER BY DT"
    
    with sql_engine.connect() as conn:
        result = pd.read_sql(text(sql), conn, params=params)
    
    return result

print("规模趋势查询函数定义完成")

---

### 数据处理完成

**已完成的处理步骤**:
1. 获取待处理日期列表
2. 执行14个维度的规模统计查询
3. 清理临时数据
4. 数据验证

---

**下一章**: [5_策略实现与回测](5_策略实现与回测.ipynb)